<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/04-RAG_with_VectorStore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [ ]:
!pip install -q openai==1.107.0 chromadb==1.0.21 google-genai==1.36.0 llama-index==0.14.0 llama-index-llms-google-genai==0.3.0 langchain==0.3.27 \
                llama-index-vector-stores-chroma==0.5.2 langchain-chroma==0.2.5 langchain-openai==0.3.32 langchain-google-genai==2.0.10 \
                jedi==0.19.2

In [ ]:
import os
# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_API_KEY>"
# os.environ["GOOGLE_API_KEY"] = "<YOUR_API_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
    os.environ["GOOGLE_API_KEY"] = userdata.get('Google_api_key')
except ImportError:
    pass  # Running locally — keys are expected in the environment

# Load the Dataset (CSV)


## Download


The dataset includes several articles from the TowardsAI blog, which provide an in-depth explanation of the LLaMA2 model. Read the dataset as a long string.


In [ ]:
!curl -o ./mini-dataset.csv https://raw.githubusercontent.com/AlaFalaki/tutorial_notebooks/main/data/mini-llama-articles.csv

## Read File


In [ ]:
import csv

text = ""

# Load the file as a JSON
with open("./mini-dataset.csv", mode="r", encoding="utf-8") as file:
    csv_reader = csv.reader(file)

    for idx, row in enumerate(csv_reader):
        if idx == 0:
            continue
        text += row[1]

# The number of characters in the dataset.
print(len(text))

# Chunking


In [ ]:
chunk_size = 512
chunks = []

# Split the long text into smaller manageable chunks of 512 characters.
for i in range(0, len(text), chunk_size):
    chunks.append(text[i : i + chunk_size])

print(len(chunks))

#Interface of Chroma with LlamaIndex


In [ ]:
from llama_index.core import Document

# Convert the chunks to Document objects so the LlamaIndex framework can process them.
documents = [Document(text=t) for t in chunks]

## Save on Chroma


In [ ]:
import chromadb

# create client and a new collection
# chromadb.EphemeralClient saves data in-memory.
chroma_client = chromadb.PersistentClient(path="./mini-chunked-dataset")
chroma_collection = chroma_client.create_collection("mini-chunked-dataset")

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

# Define a storage context object using the created vector database.
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.openai import OpenAIEmbedding

# Build index / generate embeddings using OpenAI embedding model
index = VectorStoreIndex.from_documents(
    documents,
    embed_model=OpenAIEmbedding(model="text-embedding-3-small"),
    storage_context=storage_context,
    show_progress=True,
)

## Query Dataset


In [ ]:
# Define a query engine that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.

from llama_index.llms.google_genai import GoogleGenAI

llm = GoogleGenAI(model="gemini-2.5-flash",max_tokens=512, temperature=1)

query_engine = index.as_query_engine(llm=llm, similarity_top_k=5)

In [ ]:
response = query_engine.query("How many parameters LLaMA2 model has?")
print(response)

# Interface of Chroma with LangChain


In [ ]:
from langchain.schema.document import Document

# Convert the chunks to Document objects so the LangChain framework can process them.
documents = [Document(page_content=t) for t in chunks]

## Save on Chroma


In [ ]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# Add the documents to chroma DB and create Index / embeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

chroma_db = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory="./mini-chunked-dataset",
    collection_name="mini-chunked-dataset",
)

## Query Dataset


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Initializing the LLM model
#llm = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=512)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_tokens=512,
)

In [ ]:
from langchain.chains import RetrievalQA

query = "How many parameters LLaMA 2 model has?"

retriever = chroma_db.as_retriever(search_kwargs={"k": 4})
# Define a RetrievalQA chain that is responsible for retrieving related pieces of text,
# and using a LLM to formulate the final answer.
chain = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever)

response = chain.invoke(query)
print(response["result"])

## Optional: ChromaDB Collection Inspector

Inspect what is stored in your ChromaDB collection — view document count and
a sample of the stored chunks with their metadata.

In [ ]:
# Inspect the ChromaDB collection
count = chroma_collection.count()
print(f"Total documents in collection: {count}")

# Sample the first few stored documents
sample = chroma_collection.get(limit=3, include=["documents", "metadatas"])
print("\nSample chunks from ChromaDB:")
for i, (doc, meta) in enumerate(zip(sample["documents"], sample["metadatas"])):
    print(f"\n--- Chunk {i+1} ---")
    print(f"Text (first 200 chars): {doc[:200]}")
    print(f"Metadata: {meta}")

## Optional: LlamaIndex vs. LangChain Comparison

Both LlamaIndex and LangChain can query the same ChromaDB collection.
Compare their answers to the same question side by side.

In [ ]:
compare_query = "What are the safety features of LLaMA 2?"

# LlamaIndex answer
li_response = query_engine.query(compare_query)
print("=== LlamaIndex Answer ===")
print(str(li_response)[:500])
print()

# LangChain answer
lc_response = chain.invoke(compare_query)
print("=== LangChain Answer ===")
print(lc_response["result"][:500])